In [1]:
#export

import smtplib
import ssl
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.header import Header
from email.utils import formataddr, formatdate

import logs 
import logging 

tag="mail_manager"



In [1]:
#export 


def _normalize_recipients(recipients):
    """
    Acepta str con separadores (',' o ';') o lista/tupla.
    Devuelve (lista_recipientes, header_to_str)
    """
    if recipients is None:
        return [], ""

    if isinstance(recipients, (list, tuple, set)):
        recip_list = [str(r).strip() for r in recipients if str(r).strip()]
    else:
        # str: permitir comas o puntos y coma como separadores
        parts = str(recipients).replace(";", ",").split(",")
        recip_list = [p.strip() for p in parts if p.strip()]

    # Cadena para el header "To" (nombre opcional podría ir con formataddr)
    to_header = ", ".join(recip_list)
    return recip_list, to_header

def send_email(subject, body, to_email, log=None, cc=None, bcc=None):
    tag = "mail_manager"
    method = "send_email"

    logger, handler = logs.crea_log(log, method)  

    try:
        from_email = "automatizaciondevif@gmail.com"
        smtp_server = "smtp.gmail.com"
        smtp_port = 587
        smtp_user = "automatizaciondevif@gmail.com"
        smtp_password = "fwwfbuvufhooykkc"  # App Password de Gmail (2FA)

        # Normalizar destinatarios
        to_list, to_header = _normalize_recipients(to_email)
        cc_list, cc_header = _normalize_recipients(cc)
        bcc_list, _ = _normalize_recipients(bcc)

        if not to_list and not cc_list and not bcc_list:
            raise ValueError("No hay destinatarios (To/CC/BCC).")

        # Todos los destinatarios efectivos para SMTP
        all_rcpt = to_list + cc_list + bcc_list

        # Crear el mensaje
        msg = MIMEMultipart()
        # From puede incluir nombre legible si querés: formataddr(("Automatización", from_email))
        msg["From"] = from_email
        if to_header:
            msg["To"] = to_header
        if cc_header:
            msg["Cc"] = cc_header

        # Asunto y fecha con codificación segura
        msg["Subject"] = str(Header(subject, "utf-8"))
        msg["Date"] = formatdate(localtime=True)

        # Cuerpo (HTML) en UTF-8
        msg.attach(MIMEText(body, "html", "utf-8"))

        # Envío
        context = ssl.create_default_context()
        with smtplib.SMTP(smtp_server, smtp_port) as server:
            server.ehlo()
            server.starttls(context=context)
            server.ehlo()
            server.login(smtp_user, smtp_password)
            server.sendmail(from_email, all_rcpt, msg.as_string())

        if log:
            logger.debug(f"[{tag}]-[{method}]-OK Correo enviado a: {all_rcpt}")

    except Exception as e:
        logger.error(f"[{tag}]-[{method}]-ERROR {e}.")
    finally:
        logs.cierra_log(logger, handler)


In [ ]:
###################################################################
###################################################################
###################################################################

In [17]:

subject = "Listado de Videos por Canal"
to_email = "zzddfge@gmail.com"


#body = generate_email_body(canales)
body="Prueba"
send_email(subject, body, to_email, log=logging.DEBUG)

2025-08-09 00:49:25,111 - DEBUG - [mail_manager]-send_email-START Correo enviado exitosamente.
2025-08-09 00:49:25,111 - DEBUG - [mail_manager]-send_email-START Correo enviado exitosamente.
2025-08-09 00:49:25,111 - DEBUG - [mail_manager]-send_email-START Correo enviado exitosamente.
